# GitHub-backed multi-layout invoice dataset
Select a **T4 GPU** runtime. Run cells 1-6, then pause to verify source PDFs in the dashboard; run cell 7 after each review batch. Do not use Run all before reviewing labels. PDFs are already public in this repository; do not upload them again. Verified labels will also be **public** in `public_invoice_labels/` after you run the push cell. Draft images/exports stay temporary in Colab. Before cell 7, add a fine-grained `GITHUB_TOKEN` with Contents: Read and write for this repository to Colab Secrets and enable Notebook access. Never paste the token into a notebook or chat.

In [ ]:
#@title 1. Configure GitHub-only public labels
WORK_DIR = '/content/invoice_ocr_work'
VERIFIED_BY = '' #@param {type:'string'}
import pathlib
print('Temporary Colab workspace:', WORK_DIR)
print('Verified JSON labels will be pushed to the PUBLIC GitHub repo. Token is needed only at cell 7.')

In [ ]:
#@title 2. Clone the current public repo and its PDFs/verified labels
import os, pathlib, subprocess
PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ubaid-148/OCR.git',str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR/'.git').is_dir():
    subprocess.run(['git','-C',str(PROJECT_DIR),'pull','--ff-only'], check=True)
else: raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh Colab runtime.')
os.chdir(PROJECT_DIR)
PDF_DIR = str(PROJECT_DIR/'public_invoice_pdfs')
LABEL_DIR = str(PROJECT_DIR/'public_invoice_labels')
if not pathlib.Path(PDF_DIR).is_dir(): raise ValueError('Public PDFs are missing from the Git clone')
print('Code ready at', PROJECT_DIR)
print('PDFs:', PDF_DIR, 'Public labels:', LABEL_DIR)

In [ ]:
#@title 3. Install isolated GPU OCR environment
import os, pathlib, shutil, subprocess, sys
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi','-L'], capture_output=True).returncode == 0
if not gpu_runtime:
    raise RuntimeError('GPU runtime required: Runtime > Change runtime type > T4 GPU')
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable,'-m','venv','--without-pip',str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR/'bin'/'python')
ocr_pip = [sys.executable,'-m','pip','--python',OCR_PYTHON]
subprocess.run([*ocr_pip,'install','-q','--upgrade','pip'], check=True)
subprocess.run([*ocr_pip,'install','-q','torch==2.9.1+cpu','--index-url','https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([*ocr_pip,'install','-q','paddleocr==3.7.0','pypdfium2==5.13.0'], check=True)
subprocess.run([*ocr_pip,'uninstall','-y','paddlepaddle','paddlepaddle-gpu'], check=True)
subprocess.run([*ocr_pip,'install','-q','paddlepaddle-gpu==3.3.1','-i','https://www.paddlepaddle.org.cn/packages/stable/cu126/'], check=True)
OCR_ENV = os.environ.copy()
OCR_ENV.update(OCR_DEVICE='gpu:0', OCR_TARGETED_RETRY='true', USE_LOCAL_AI='false', FLAGS_use_mkldnn='0')
subprocess.run([OCR_PYTHON,'-u',str(PROJECT_DIR/'check_ocr_runtime.py')], cwd=PROJECT_DIR, env=OCR_ENV, check=True)
print('OCR drafting runtime ready.')

In [ ]:
#@title 4. Render pages and initialize resumable annotations
import subprocess
subprocess.run([OCR_PYTHON,'-m','training.invoice_dataset','prepare','--pdf-dir',PDF_DIR,'--work-dir',WORK_DIR,'--labels-dir',LABEL_DIR,'--dpi','200','--allow-public-pdf-dir','--allow-public-labels'], cwd=PROJECT_DIR, env=OCR_ENV, check=True)

In [ ]:
#@title 5. Generate OCR drafts for all PDFs (resumable; may take several minutes)
subprocess.run([OCR_PYTHON,'-u','-m','training.invoice_dataset','draft','--work-dir',WORK_DIR,'--languages','ara+eng'], cwd=PROJECT_DIR, env=OCR_ENV, check=True)
print('Drafts are not ground truth. Verify them below against every source page.')

In [ ]:
#@title 6. Review and verify labels against the PDFs
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from training.colab_annotator import launch
dashboard = launch(WORK_DIR, VERIFIED_BY)
print('PAUSE: verify source pages in the dashboard. After a batch, run cell 7 to push labels to GitHub. Training needs 80+ verified/included labels.')

In [ ]:
#@title 7. Publish verified labels to GitHub (rerun after each review batch)
import json
from google.colab import userdata
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as error:
    raise RuntimeError('To push labels: add GITHUB_TOKEN in Colab Secrets and enable Notebook access. Do not paste the token into a cell.') from error
if not GITHUB_TOKEN:
    raise ValueError('GITHUB_TOKEN is empty in Colab Secrets. Add a token with Contents: Read and write for ubaid-148/OCR.')
from training.github_sync import push_verified_labels
push_result = push_verified_labels(PROJECT_DIR, pathlib.Path(WORK_DIR), GITHUB_TOKEN)
print(json.dumps(push_result, indent=2))

In [ ]:
#@title 8. Validate annotation progress
from training.invoice_dataset import validation_report
import json
report = validation_report(pathlib.Path(WORK_DIR))
print(json.dumps(report, ensure_ascii=False, indent=2))
if report['errors']:
    raise ValueError('Fix annotation errors before export.')

In [ ]:
#@title 9. Optional local export preview; training notebook exports again from GitHub
MIN_VERIFIED = 80 #@param {type:'integer'}
from training.invoice_dataset import export_qwen
summary = export_qwen(pathlib.Path(WORK_DIR), min_verified=MIN_VERIFIED)
print(json.dumps(summary, indent=2))
print('Temporary SFT files:', pathlib.Path(WORK_DIR)/'exports')

After all required labels are verified, run cell 7 and confirm `pushed: true` (or no new labels). Then start a **fresh GPU runtime** and open `colab_train.ipynb`; it recreates page images/exports from the public PDFs and committed labels. No Drive mount is required.